### 1) Categorical features
- City (3 categories)
- Gender (2 categories)
- EverBenched (2 categories)

### 2) Features to transform into interval pattern structures
- Education
- JoiningYear
- PaymentTier
- Age
- ExperienceInCurrentDomain

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
data = pd.read_csv('Employee.csv')
data.head()

,Education,JoiningYear,City,PaymentTier,Age,Gender,EverBenched,ExperienceInCurrentDomain,LeaveOrNot
0,Bachelors,2017,Bangalore,3,34,Male,No,0,0
1,Bachelors,2013,Pune,1,28,Female,No,3,1
2,Bachelors,2014,New Delhi,3,38,Female,No,2,0
3,Masters,2016,Bangalore,3,27,Male,No,5,1
4,Masters,2017,Pune,3,24,Male,Yes,2,1


In [3]:
#deleting diplicates with different values of target
duplicates_with_diff_leave = data[data.duplicated(subset=data.columns.difference(['LeaveOrNot']), keep=False)]
result = duplicates_with_diff_leave.groupby(list(data.columns.difference(['LeaveOrNot']))).filter(lambda x: x['LeaveOrNot'].nunique() > 1)
duplicate_indices = result.index
cleaned_data = data.drop(duplicate_indices)

In [4]:
#Categorical features encoding

#City
cleaned_data['city_is_bangalore'] = (cleaned_data.City == 'Bangalore').astype(int)
cleaned_data['city_is_pune'] = (cleaned_data.City == 'Pune').astype(int)
cleaned_data['city_is_new_delhi'] = (cleaned_data.City == 'New Delhi').astype(int)

#Gender
cleaned_data['gender_is_male'] = (cleaned_data.Gender == 'Male').astype(int)
cleaned_data['gender_is_female'] = (cleaned_data.Gender == 'Female').astype(int)

#EverBenched
cleaned_data['is_ever_benched'] = (cleaned_data.EverBenched == 'Yes').astype(int)
cleaned_data['is_never_benched'] = (cleaned_data.EverBenched == 'No').astype(int)

In [5]:
#Drop already unnecessary categorical features
cleaned_encoded_data = cleaned_data.drop(columns = ['City', 'Gender', 'EverBenched'])
cleaned_encoded_data.head()

,Education,JoiningYear,PaymentTier,Age,ExperienceInCurrentDomain,LeaveOrNot,city_is_bangalore,city_is_pune,city_is_new_delhi,gender_is_male,gender_is_female,is_ever_benched,is_never_benched
0,Bachelors,2017,3,34,0,0,1,0,0,1,0,0,1
1,Bachelors,2013,1,28,3,1,0,1,0,0,1,0,1
2,Bachelors,2014,3,38,2,0,0,0,1,0,1,0,1
3,Masters,2016,3,27,5,1,1,0,0,1,0,0,1
4,Masters,2017,3,24,2,1,0,1,0,1,0,1,0


I will code every interval value as a tuple (value, value) where the value is 0 for no degree (does not really exist in practice), 1 for Bachelor, 2 for Master, 3 for PHD.

In [6]:
data = cleaned_encoded_data
education_coded = []
education_categ_coded = []
for row in data.itertuples(index=False):
    if row.Education == 'Bachelors':
        education_coded.append((1,1))
        education_categ_coded.append(1)
    elif row.Education == 'Masters':
        education_coded.append((2,2))
        education_categ_coded.append(2)
    elif row.Education == 'PHD':
        education_coded.append((3,3))
        education_categ_coded.append(3)
    else:
        education_coded.append((0,0))
        education_categ_coded.append(0)
data['Education_intervals'] = education_coded
data['Education_categories'] = education_categ_coded
# I will 
# data.drop('Education', axis=1, inplace=True)
data.head()

,Education,JoiningYear,PaymentTier,Age,ExperienceInCurrentDomain,LeaveOrNot,city_is_bangalore,city_is_pune,city_is_new_delhi,gender_is_male,gender_is_female,is_ever_benched,is_never_benched,Education_intervals,Education_categories
0,Bachelors,2017,3,34,0,0,1,0,0,1,0,0,1,"(1, 1)",1
1,Bachelors,2013,1,28,3,1,0,1,0,0,1,0,1,"(1, 1)",1
2,Bachelors,2014,3,38,2,0,0,0,1,0,1,0,1,"(1, 1)",1
3,Masters,2016,3,27,5,1,1,0,0,1,0,0,1,"(2, 2)",2
4,Masters,2017,3,24,2,1,0,1,0,1,0,1,0,"(2, 2)",2


In [7]:
joining_year_coded = []
joining_year_categ_coded = []
# for normalization
min_year = min(list(data['JoiningYear']))
for row in data.itertuples(index=False):
    joining_year_coded.append((row.JoiningYear - min_year, row.JoiningYear - min_year))
    joining_year_categ_coded.append(row.JoiningYear - min_year)
data['JoiningYear_intervals'] = joining_year_coded
data['JoiningYear_categories'] = joining_year_categ_coded
# data.drop('JoiningYear', axis=1, inplace=True)
data.head()

,Education,JoiningYear,PaymentTier,Age,ExperienceInCurrentDomain,LeaveOrNot,city_is_bangalore,city_is_pune,city_is_new_delhi,gender_is_male,gender_is_female,is_ever_benched,is_never_benched,Education_intervals,Education_categories,JoiningYear_intervals,JoiningYear_categories
0,Bachelors,2017,3,34,0,0,1,0,0,1,0,0,1,"(1, 1)",1,"(5, 5)",5
1,Bachelors,2013,1,28,3,1,0,1,0,0,1,0,1,"(1, 1)",1,"(1, 1)",1
2,Bachelors,2014,3,38,2,0,0,0,1,0,1,0,1,"(1, 1)",1,"(2, 2)",2
3,Masters,2016,3,27,5,1,1,0,0,1,0,0,1,"(2, 2)",2,"(4, 4)",4
4,Masters,2017,3,24,2,1,0,1,0,1,0,1,0,"(2, 2)",2,"(5, 5)",5


In [8]:
payment_tier_coded = []
payment_tier_categ_coded = []
for row in data.itertuples(index=False):
    payment_tier_coded.append((row.PaymentTier, row.PaymentTier))
    payment_tier_categ_coded.append(row.PaymentTier)
data['PaymentTier_intervals'] = payment_tier_coded
data['PaymentTier_categories'] = payment_tier_categ_coded
data.head()

,Education,JoiningYear,PaymentTier,Age,ExperienceInCurrentDomain,LeaveOrNot,city_is_bangalore,city_is_pune,city_is_new_delhi,gender_is_male,gender_is_female,is_ever_benched,is_never_benched,Education_intervals,Education_categories,JoiningYear_intervals,JoiningYear_categories,PaymentTier_intervals,PaymentTier_categories
0,Bachelors,2017,3,34,0,0,1,0,0,1,0,0,1,"(1, 1)",1,"(5, 5)",5,"(3, 3)",3
1,Bachelors,2013,1,28,3,1,0,1,0,0,1,0,1,"(1, 1)",1,"(1, 1)",1,"(1, 1)",1
2,Bachelors,2014,3,38,2,0,0,0,1,0,1,0,1,"(1, 1)",1,"(2, 2)",2,"(3, 3)",3
3,Masters,2016,3,27,5,1,1,0,0,1,0,0,1,"(2, 2)",2,"(4, 4)",4,"(3, 3)",3
4,Masters,2017,3,24,2,1,0,1,0,1,0,1,0,"(2, 2)",2,"(5, 5)",5,"(3, 3)",3


In [9]:
age_coded = []
age_categ_coded = []
# for normalization
min_age = min(list(data['Age']))
for row in data.itertuples(index=False):
    age_coded.append((row.Age - min_age, row.Age - min_age))
    age_categ_coded.append(row.Age - min_age)
data['Age_intervals'] = age_coded
data['Age_categories'] = age_categ_coded
data.head()

,Education,JoiningYear,PaymentTier,Age,ExperienceInCurrentDomain,LeaveOrNot,city_is_bangalore,city_is_pune,city_is_new_delhi,gender_is_male,...,is_ever_benched,is_never_benched,Education_intervals,Education_categories,JoiningYear_intervals,JoiningYear_categories,PaymentTier_intervals,PaymentTier_categories,Age_intervals,Age_categories
0,Bachelors,2017,3,34,0,0,1,0,0,1,...,0,1,"(1, 1)",1,"(5, 5)",5,"(3, 3)",3,"(12, 12)",12
1,Bachelors,2013,1,28,3,1,0,1,0,0,...,0,1,"(1, 1)",1,"(1, 1)",1,"(1, 1)",1,"(6, 6)",6
2,Bachelors,2014,3,38,2,0,0,0,1,0,...,0,1,"(1, 1)",1,"(2, 2)",2,"(3, 3)",3,"(16, 16)",16
3,Masters,2016,3,27,5,1,1,0,0,1,...,0,1,"(2, 2)",2,"(4, 4)",4,"(3, 3)",3,"(5, 5)",5
4,Masters,2017,3,24,2,1,0,1,0,1,...,1,0,"(2, 2)",2,"(5, 5)",5,"(3, 3)",3,"(2, 2)",2


In [10]:
# I will code experience as intervals and also will delete experience levels "6" and "7". I will concider them as 5 (5+), because these classes are very small
experience_coded = []
experience_categ_coded = []
for row in data.itertuples(index=False):
    if row.ExperienceInCurrentDomain < 6:
        experience_coded.append((row.ExperienceInCurrentDomain, row.ExperienceInCurrentDomain))
        experience_categ_coded.append(row.ExperienceInCurrentDomain)
    else:
        experience_coded.append((5, 5))
        experience_categ_coded.append(5)
data["ExperienceInCurrentDomain_intervals"] = experience_coded
data['ExperienceInCurrentDomain_categories'] = experience_categ_coded
data.head()

,Education,JoiningYear,PaymentTier,Age,ExperienceInCurrentDomain,LeaveOrNot,city_is_bangalore,city_is_pune,city_is_new_delhi,gender_is_male,...,Education_intervals,Education_categories,JoiningYear_intervals,JoiningYear_categories,PaymentTier_intervals,PaymentTier_categories,Age_intervals,Age_categories,ExperienceInCurrentDomain_intervals,ExperienceInCurrentDomain_categories
0,Bachelors,2017,3,34,0,0,1,0,0,1,...,"(1, 1)",1,"(5, 5)",5,"(3, 3)",3,"(12, 12)",12,"(0, 0)",0
1,Bachelors,2013,1,28,3,1,0,1,0,0,...,"(1, 1)",1,"(1, 1)",1,"(1, 1)",1,"(6, 6)",6,"(3, 3)",3
2,Bachelors,2014,3,38,2,0,0,0,1,0,...,"(1, 1)",1,"(2, 2)",2,"(3, 3)",3,"(16, 16)",16,"(2, 2)",2
3,Masters,2016,3,27,5,1,1,0,0,1,...,"(2, 2)",2,"(4, 4)",4,"(3, 3)",3,"(5, 5)",5,"(5, 5)",5
4,Masters,2017,3,24,2,1,0,1,0,1,...,"(2, 2)",2,"(5, 5)",5,"(3, 3)",3,"(2, 2)",2,"(2, 2)",2


In [11]:
data.to_csv("cleaned_encoded_data.csv", index=False)

In [26]:
print(data.iloc[14])

Education                               Bachelors
JoiningYear                                  2017
PaymentTier                                     1
Age                                            29
ExperienceInCurrentDomain                       3
LeaveOrNot                                      0
city_is_bangalore                               1
city_is_pune                                    0
city_is_new_delhi                               0
gender_is_male                                  1
gender_is_female                                0
is_ever_benched                                 0
is_never_benched                                1
Education_intervals                        (1, 1)
Education_categories                            1
JoiningYear_intervals                      (5, 5)
JoiningYear_categories                          5
PaymentTier_intervals                      (1, 1)
PaymentTier_categories                          1
Age_intervals                              (7, 7)


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=729b1cc4-35f2-4c75-899d-e722545e10c4' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>